#Mount Drive + Imports

In [ ]:
from pyspark.sql import SparkSession
from google.colab import drive
drive.mount('/content/drive')

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

BASE = "/content/drive/MyDrive/CourseWork/CSV_Files/Cleaned_Output"

BUSINESS_PATH = f"{BASE}/Cleaned_business.csv"
EVENTS_PATH   = f"{BASE}/Cleaned_checkin_events.csv"

df_biz = spark.read.option("header", True).option("inferSchema", True).csv(BUSINESS_PATH)
df_evt = spark.read.option("header", True).option("inferSchema", True).csv(EVENTS_PATH)

df_biz.createOrReplaceTempView("cleaned_business")
df_evt.createOrReplaceTempView("cleaned_checkin_events")

print("Views created: cleaned_business, cleaned_checkin_events")
print("Rows business:", df_biz.count(), " | Rows events:", df_evt.count())

Mounted at /content/drive
Views created: cleaned_business, cleaned_checkin_events
Rows business: 150346  | Rows events: 13356875


#Normalize Views

In [ ]:
# Normalized views: robust trimming/case + safe casting
spark.sql("""
CREATE OR REPLACE TEMP VIEW biz_norm AS
SELECT
  business_id,
  name,
  LOWER(TRIM(city))  AS city,
  LOWER(TRIM(state)) AS state,
  CAST(TRIM(is_open) AS INT) AS is_open,
  CAST(stars AS DOUBLE) AS stars,
  CAST(review_count AS INT) AS review_count,
  LOWER(TRIM(primary_category)) AS primary_category,
  COALESCE(NULLIF(TRIM(price_range_label), ''), 'Unknown') AS price_range_label,
  CAST(review_count_outlier_flag AS INT) AS review_count_outlier_flag
FROM cleaned_business
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW evt_norm AS
SELECT
  business_id,
  checkin_ts,
  checkin_date,
  CAST(hour AS INT) AS hour,
  CAST(dow  AS INT) AS dow,
  CAST(is_weekend AS INT) AS is_weekend
FROM cleaned_checkin_events
""")

print("Normalized views created: biz_norm, evt_norm")


Normalized views created: biz_norm, evt_norm


#Finding the State with the Maximum Businesses

In [ ]:
print("\n---------FINDING: Top state by number of businesses ---------")

top_state_df = spark.sql("""
SELECT
  state,
  COUNT(DISTINCT business_id) AS business_count
FROM biz_norm
WHERE state IS NOT NULL AND state <> ''
GROUP BY state
ORDER BY business_count DESC
LIMIT 1
""")

top_state_df.show(truncate=False)

TOP_STATE = top_state_df.collect()[0]["state"]
print("TOP_STATE =", TOP_STATE)



---------FINDING: Top state by number of businesses ---------
+-----+--------------+
|state|business_count|
+-----+--------------+
|pa   |34039         |
+-----+--------------+

TOP_STATE = pa


#Q1: Raw joined events sample (shows business + event columns)

In [ ]:
print(f"\n---------Q1: Raw joined events sample for {TOP_STATE} -----------")
spark.sql(f"""
SELECT
  b.business_id, b.name, b.city, b.state, b.is_open, b.primary_category, b.stars,
  e.checkin_date, e.checkin_ts, e.hour, e.dow, e.is_weekend
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
ORDER BY e.checkin_date DESC, e.hour DESC
LIMIT 20
""").show(20, truncate=False)


---------Q1: Raw joined events sample for pa -----------
+----------------------+----------------------------------------+-------------------+-----+-------+----------------------------+-----+------------+-------------------+----+---+----------+
|business_id           |name                                    |city               |state|is_open|primary_category            |stars|checkin_date|checkin_ts         |hour|dow|is_weekend|
+----------------------+----------------------------------------+-------------------+-----+-------+----------------------------+-----+------------+-------------------+----+---+----------+
|Sv1MEZP-mMfp8SmE0hwYEA|Terakawa Ramen                          |philadelphia       |pa   |1      |japanese                    |4.5  |2022-01-19  |2022-01-19 16:23:04|16  |4  |0         |
|stAPGCcWDJkNWA5u3nqa8Q|Tacotacomexican                         |philadelphia       |pa   |1      |mexican                     |4.5  |2022-01-19  |2022-01-19 16:08:26|16  |4  |0         |
|9

#Q2: Top 20 businesses by total check-ins in the Top State

In [ ]:
print(f"\n----------- Q2: Top 20 businesses by TOTAL check-ins in {TOP_STATE} --------------")
spark.sql(f"""
SELECT
  b.business_id, b.name, b.city, b.state, b.primary_category, b.is_open, b.stars,
  COUNT(*) AS total_checkins
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
GROUP BY b.business_id, b.name, b.city, b.state, b.primary_category, b.is_open, b.stars
ORDER BY total_checkins DESC
LIMIT 20
""").show(20, truncate=False)


----------- Q2: Top 20 businesses by TOTAL check-ins in pa --------------
+----------------------+----------------------------------------+---------------+-----+-------------------------+-------+-----+--------------+
|business_id           |name                                    |city           |state|primary_category         |is_open|stars|total_checkins|
+----------------------+----------------------------------------+---------------+-----+-------------------------+-------+-----+--------------+
|-QI8Qi8XWH3D8y8ethnajA|Philadelphia International Airport - PHL|philadelphia   |pa   |professional services    |1      |2.5  |52144         |
|ytynqOUb3hjKeJfRj5Tshw|Reading Terminal Market                 |philadelphia   |pa   |candy stores             |1      |4.5  |18615         |
|Xq-8-I0U8Artr7d70SjX-g|30th Street Station                     |philadelphia   |pa   |public transportation    |1      |4.0  |10063         |
|PP3BBaVxZLcJU54uP_wL6Q|Pat's King of Steaks                    |ph

#Q3: Weekend-share leaders (weekend-driven businesses) in the Top State

In [ ]:
print(f"\n------------- Q3: Weekend-share leaders in {TOP_STATE} (min 100 check-ins) --------------")

spark.sql(f"""
SELECT
  b.business_id, b.name, b.city, b.state, b.primary_category, b.stars,
  COUNT(*) AS total_checkins,
  SUM(CASE WHEN e.is_weekend=1 THEN 1 ELSE 0 END) AS weekend_checkins,
  ROUND(SUM(CASE WHEN e.is_weekend=1 THEN 1 ELSE 0 END) / COUNT(*), 3) AS weekend_share
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
GROUP BY b.business_id, b.name, b.city, b.state, b.primary_category, b.stars
HAVING COUNT(*) >= 100
ORDER BY weekend_share DESC, total_checkins DESC
LIMIT 20
""").show(20, truncate=False)



------------- Q3: Weekend-share leaders in pa (min 100 check-ins) --------------
+----------------------+------------------------------------+-----------------+-----+-----------------------+-----+--------------+----------------+-------------+
|business_id           |name                                |city             |state|primary_category       |stars|total_checkins|weekend_checkins|weekend_share|
+----------------------+------------------------------------+-----------------+-----+-----------------------+-----+--------------+----------------+-------------+
|O6VNi891zi38OSs89Ikvgg|Lansdowne Farmers Market            |lansdowne        |pa   |farmers market         |4.5  |106           |104             |0.981        |
|p6hPjTEtsFU7kaiUIJm2tQ|Epic Church Center City             |philadelphia     |pa   |religious organizations|4.5  |162           |158             |0.975        |
|IHCSJHLyz2MgXpJAVssC3Q|Headhouse Farmers' Market           |philadelphia     |pa   |food                   

#Q4: Peak hour per business (busiest hour) in the Top State

In [ ]:
print(f"\n------------Q4: Peak hour per business in {TOP_STATE}-----------------")

spark.sql(f"""
WITH hour_counts AS (
  SELECT
    b.business_id, b.name, b.city, b.state, b.primary_category,
    e.hour, COUNT(*) AS hour_checkins
  FROM biz_norm b
  JOIN evt_norm e ON b.business_id = e.business_id
  WHERE b.state = '{TOP_STATE}'
  GROUP BY b.business_id, b.name, b.city, b.state, b.primary_category, e.hour
),
ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY business_id ORDER BY hour_checkins DESC) AS rn
  FROM hour_counts
)
SELECT
  business_id, name, city, state, primary_category,
  hour AS peak_hour, hour_checkins
FROM ranked
WHERE rn = 1
ORDER BY hour_checkins DESC
LIMIT 20
""").show(20, truncate=False)



------------Q4: Peak hour per business in pa-----------------
+----------------------+----------------------------------------+------------+-----+----------------------+---------+-------------+
|business_id           |name                                    |city        |state|primary_category      |peak_hour|hour_checkins|
+----------------------+----------------------------------------+------------+-----+----------------------+---------+-------------+
|-QI8Qi8XWH3D8y8ethnajA|Philadelphia International Airport - PHL|philadelphia|pa   |professional services |20       |4186         |
|ytynqOUb3hjKeJfRj5Tshw|Reading Terminal Market                 |philadelphia|pa   |candy stores          |17       |2743         |
|gGyqnAlpFrka_qzpO7j4lQ|Citizens Bank Park                      |philadelphia|pa   |active life           |23       |1257         |
|IWHdx0NhDKADkGOgXgOFKQ|Tommy DiNic's                           |philadelphia|pa   |restaurants           |17       |922          |
|ITOkOiX-hfsa


#Q5: Busy but low-rated (reputation risk) in the Top State

In [ ]:
print(f"\n----------Q5: Busy but low-rated (risk watchlist) in {TOP_STATE}---------------")

spark.sql(f"""
SELECT
  b.business_id, b.name, b.city, b.state, b.primary_category,
  b.stars, b.review_count,
  COUNT(*) AS total_checkins
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
GROUP BY b.business_id, b.name, b.city, b.state, b.primary_category, b.stars, b.review_count
HAVING COUNT(*) >= 150 AND b.stars <= 3.5
ORDER BY total_checkins DESC
LIMIT 20
""").show(20, truncate=False)



----------Q5: Busy but low-rated (risk watchlist) in pa---------------
+----------------------+----------------------------------------+------------+-----+----------------------+-----+------------+--------------+
|business_id           |name                                    |city        |state|primary_category      |stars|review_count|total_checkins|
+----------------------+----------------------------------------+------------+-----+----------------------+-----+------------+--------------+
|-QI8Qi8XWH3D8y8ethnajA|Philadelphia International Airport - PHL|philadelphia|pa   |professional services |2.5  |78          |52144         |
|PP3BBaVxZLcJU54uP_wL6Q|Pat's King of Steaks                    |philadelphia|pa   |italian               |3.0  |78          |7264          |
|sTPueJEwcRDj7ZJmG7okYA|Jim's South St                          |philadelphia|pa   |bars                  |3.5  |78          |5492          |
|IkY2ticzHEn4QFn8hQLSWg|Geno's Steaks                           |philadelphi

#Q6: High-rated but low-traffic (hidden gems) in the Top State

In [ ]:
print(f"\n------------ Q6: High-rated but low-traffic (hidden gems) in {TOP_STATE} -------------")

spark.sql(f"""
SELECT
  b.business_id, b.name, b.city, b.state, b.primary_category,
  b.stars, b.review_count,
  COUNT(*) AS total_checkins
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
GROUP BY b.business_id, b.name, b.city, b.state, b.primary_category, b.stars, b.review_count
HAVING b.stars >= 4.5 AND COUNT(*) <= 30
ORDER BY b.stars DESC, b.review_count DESC
LIMIT 20
""").show(20, truncate=False)



------------ Q6: High-rated but low-traffic (hidden gems) in pa -------------
+----------------------+-----------------------------------+------------+-----+-------------------------+-----+------------+--------------+
|business_id           |name                               |city        |state|primary_category         |stars|review_count|total_checkins|
+----------------------+-----------------------------------+------------+-----+-------------------------+-----+------------+--------------+
|4caDWodT3H9v4ekwCsh6oQ|Sammy’s Bullfrog Cafe              |harleysville|pa   |breakfast & brunch       |5.0  |78          |30            |
|7T1VAYHIS3cgsDj3iS6JWQ|The Frosted Fox Cake Shop          |philadelphia|pa   |bakeries                 |5.0  |78          |21            |
|0yqpaakk5iM70s1c-L-lkA|All That Jazz Sound & Security     |philadelphia|pa   |auto security            |5.0  |78          |20            |
|89gMRtved9315_howyDEFQ|The Tailory                        |philadelphia|pa   |se

#Q7: Top categories by footfall in the Top State

In [ ]:
print(f"\n--------- Q7: Top categories by total check-ins in {TOP_STATE} -----------------")

spark.sql(f"""
SELECT
  b.state, b.city, b.primary_category,
  COUNT(*) AS total_checkins,
  COUNT(DISTINCT b.business_id) AS businesses,
  ROUND(COUNT(*) / COUNT(DISTINCT b.business_id), 2) AS avg_checkins_per_business
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
GROUP BY b.state, b.city, b.primary_category
ORDER BY total_checkins DESC
LIMIT 20
""").show(20, truncate=False)



--------- Q7: Top categories by total check-ins in pa -----------------
+-----+------------+-------------------------+--------------+----------+-------------------------+
|state|city        |primary_category         |total_checkins|businesses|avg_checkins_per_business|
+-----+------------+-------------------------+--------------+----------+-------------------------+
|pa   |philadelphia|restaurants              |283825        |1653      |171.7                    |
|pa   |philadelphia|food                     |114694        |743       |154.37                   |
|pa   |philadelphia|nightlife                |84029         |294       |285.81                   |
|pa   |philadelphia|bars                     |60941         |211       |288.82                   |
|pa   |philadelphia|professional services    |52615         |33        |1594.39                  |
|pa   |philadelphia|breakfast & brunch       |48250         |159       |303.46                   |
|pa   |philadelphia|sandwiches      

#Q8: Open vs closed comparison (footfall)

In [ ]:
print(f"\n------------ Q8: OPEN vs CLOSED comparison in {TOP_STATE} ---------------")

spark.sql(f"""
SELECT
  b.state, b.is_open,
  COUNT(DISTINCT b.business_id) AS businesses,
  COUNT(*) AS total_checkins,
  ROUND(COUNT(*) / COUNT(DISTINCT b.business_id), 2) AS avg_checkins_per_business,
  ROUND(AVG(b.stars), 2) AS avg_stars
FROM biz_norm b
JOIN evt_norm e ON b.business_id = e.business_id
WHERE b.state = '{TOP_STATE}'
GROUP BY b.state, b.is_open
ORDER BY total_checkins DESC
LIMIT 20
""").show(20, truncate=False)



------------ Q8: OPEN vs CLOSED comparison in pa ---------------
+-----+-------+----------+--------------+-------------------------+---------+
|state|is_open|businesses|total_checkins|avg_checkins_per_business|avg_stars|
+-----+-------+----------+--------------+-------------------------+---------+
|pa   |1      |22125     |2160947       |97.67                    |3.68     |
|pa   |0      |7164      |549767        |76.74                    |3.57     |
+-----+-------+----------+--------------+-------------------------+---------+

